In [7]:
import cv2
import pickle
import mediapipe as mp
import pyautogui
import numpy as np
import time
from collections import deque

from mediapipe.tasks.python import vision
from mediapipe.tasks.python import BaseOptions

import warnings
warnings.filterwarnings('ignore')

pyautogui.FAILSAFE = True   # move mouse to corner to stop if needed

# LOAD TRAINED MODEL
model = pickle.load(open("gesture_model.pkl", "rb"))
le = pickle.load(open("label_encoder.pkl", "rb"))

# MEDIAPIPE HAND LANDMARKER
base_options = BaseOptions(model_asset_path="hand_landmarker.task")
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    num_hands=1
)
landmarker = vision.HandLandmarker.create_from_options(options)

# WEBCAM & SCREEN
cap = cv2.VideoCapture(0)
screen_w, screen_h = pyautogui.size()
prev_x, prev_y = 0, 0

mouse_enabled = False
drag_active = False
last_action_time = 0
cooldown = 0.7

gesture_buffer = deque(maxlen=7)
CONFIDENCE_THRESHOLD = 0.6

def draw_landmarks(frame, hand):
    h, w, _ = frame.shape
    for lm in hand:
        cx, cy = int(lm.x * w), int(lm.y * h)
        cv2.circle(frame, (cx, cy), 5, (0, 255, 0), -1)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = landmarker.detect(mp_image)

    gesture = "NO_HAND"
    confidence = 0.0

    if result.hand_landmarks:
        hand = result.hand_landmarks[0]
        draw_landmarks(frame, hand)

        features = []
        for lm in hand:
            features.extend([lm.x, lm.y])

        if len(features) == 42:
            features = np.array(features).reshape(1, -1)

            proba = model.predict_proba(features)
            confidence = np.max(proba)

            if confidence >= CONFIDENCE_THRESHOLD:
                gesture_id = np.argmax(proba)
                gesture = le.inverse_transform([gesture_id])[0]
            else:
                gesture = "UNKNOWN"

            gesture_buffer.append(gesture)
            gesture = max(set(gesture_buffer), key=gesture_buffer.count)

            index_tip = hand[8]
            x = int(index_tip.x * screen_w)
            y = int(index_tip.y * screen_h)

            curr_x = prev_x + (x - prev_x) / 6
            curr_y = prev_y + (y - prev_y) / 6
            current_time = time.time()

            if mouse_enabled:

                if gesture == "MOVE":
                    pyautogui.moveTo(curr_x, curr_y)

                elif gesture == "LEFT_CLICK" and current_time - last_action_time > cooldown:
                    pyautogui.click()
                    last_action_time = current_time

                elif gesture == "RIGHT_CLICK" and current_time - last_action_time > cooldown:
                    pyautogui.rightClick()
                    last_action_time = current_time

                elif gesture == "DRAG" and not drag_active:
                    pyautogui.mouseDown()
                    drag_active = True

                elif gesture == "STOP" and drag_active:
                    pyautogui.mouseUp()
                    drag_active = False

                elif gesture == "SCROLL":
                    pyautogui.scroll(-30)

            prev_x, prev_y = curr_x, curr_y

    cv2.putText(frame, f"Gesture: {gesture}", (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,255), 2)

    cv2.putText(frame, f"Confidence: {confidence:.2f}", (10, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,255,0), 2)

    cv2.putText(frame,
                f"Mouse: {'ON' if mouse_enabled else 'OFF'} | P = Toggle | Q = Quit",
                (10, 120),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,0,0), 2)

    cv2.putText(frame,
            f"Buffer: {list(gesture_buffer)}",
            (10, 160),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255,255,255),
            1)

    cv2.imshow("Gesture Mouse Control", frame)

    key = cv2.waitKey(1) & 0xFF

    if key == ord('p'):
        mouse_enabled = not mouse_enabled
        time.sleep(0.3)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
landmarker.close()
